In [1]:
!pip install matplotlib==3.10.6 \
numpy==2.3.3\
pandas==2.3.3

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
tables = []

for year in range(2013, 2026):
  tables.append(pd.read_excel(f"SPK_IPOs_{year}.xlsx"))

all_ipos = pd.concat(tables, ignore_index=True)
print(f"Dataset: {all_ipos.shape[0]} IPOs, {all_ipos.shape[1]} columns")
print(all_ipos.columns.tolist())

Dataset: 260 IPOs, 17 columns
['Dönem', 'Borsa Kodu', 'Şirket Ünvanı', 'Halka Arz Şekli', 'Halka Arz Oranı (%)', 'Halka Arz Fiyatı (TL)', 'Ortak Satış Tutarı (Bin TL)', 'Nakit Sermaye Artırım Tutarı (Bin TL)', 'Satışa Hazır Bekletilen Pay Tutarı (Bin TL)', 'Satışa Sunulan Nominal Toplam Tutar (Bin TL)', 'Mevcut Sermaye (Bin TL)', 'Yeni Sermaye (Bin TL)', 'Satışa Sunulan Toplam Tutar (Piyasa Değeri Bin TL)', 'Satışa Sunulan Toplam Tutar (Piyasa Değeri Bin ABD Doları)', 'İlk İşlem Gördüğü Pazar', 'Halka Arza Aracılık Eden Kurum', 'Borsada İşlem Görme Tarihi']


In [4]:
tables_overs = []

for year in range(2020, 2024):
  tables_overs.append(pd.read_excel(f"SPK_IPOs_Overs_{year}.xlsx", header=1))

all_ipos_overs = pd.concat(tables_overs, ignore_index=True)
print(f"Dataset overs: {all_ipos_overs.shape[0]} IPOs, {all_ipos_overs.shape[1]} columns")
print(all_ipos_overs.columns.tolist())

Dataset overs: 131 IPOs, 19 columns
['Sıra              ', 'Şirket Adı', 'İzahname Onay\nKurul Karar Tarihi', 'Halka Arz Tarihi\n', 'Borsada İşlem Görme Tarihi', 'Halka Arza Aracılık Eden Yetkili Kuruluş', 'İşlem Gördüğü Pazar', 'Toplanan Fon Miktarı (TL)*', 'Toplanan Fon Miktarı (USD)', 'Mevcut Sermaye (TL)', 'Halka Arz Sonrası Sermaye (TL)', 'Halka Arz Edilen Pay (Nominal TL)', 'Nominal Sermaye Artış Tutarı (TL)', 'Nominal Ortak Satış Tutarı (TL)', 'Halka Arz Fiyatı (TL)', 'Talep Edilen Pay (Nominal TL)', 'Talebin Halka Arza Oranı', 'Dağıtımı Yapılan Yatırımcı Sayısı', 'Satış Yöntemi**']


In [5]:
print("MAIN")
print(all_ipos.head())
print()
print("OVERS")
print(all_ipos_overs.head())

MAIN
      Dönem Borsa Kodu                                      Şirket Ünvanı  \
0  2013 / 2      HLGYO            HALK GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.   
1  2013 / 4      SRVGY          SERVET GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.   
2  2013 / 4      PGSUS                     PEGASUS HAVA TAŞIMACILIĞI A.Ş.   
3  2013 / 5      ROYAL  ROYAL HALI İPLİK TEKSTİL MOBİLYA SANAYİ VE TİC...   
4  2013 / 5       ODAS           ODAŞ ELEKTRİK ÜRETİM SANAYİ TİCARET A.Ş.   

                    Halka Arz Şekli  Halka Arz Oranı (%)  \
0                  Sermaye Artırımı                28.00   
1  Sermaye Artırımı ve Ortak Satışı                25.00   
2  Sermaye Artırımı ve Ortak Satışı                35.00   
3  Sermaye Artırımı ve Ortak Satışı                28.75   
4                  Sermaye Artırımı                28.57   

   Halka Arz Fiyatı (TL)  Ortak Satış Tutarı (Bin TL)  \
0                   1.35                          0.0   
1                   2.73                      11000.0   


Clean the data and format it so that we can merge them correctly


In [6]:
all_ipos["Borsada İşlem Görme Tarihi"] = pd.to_datetime(
    all_ipos["Borsada İşlem Görme Tarihi"], format="%d.%m.%Y"
)

print(f"Rows before cleaning: {len(all_ipos)}")
print(f"Rows with missing ticker: {all_ipos['Borsa Kodu'].isna().sum()}")
print(f"Rows with missing listing date: {all_ipos['Borsada İşlem Görme Tarihi'].isna().sum()}")

Rows before cleaning: 260
Rows with missing ticker: 13
Rows with missing listing date: 13


In [7]:
all_ipos = all_ipos.dropna(subset=["Borsa Kodu"])
print(f"Rows after cleaning: {len(all_ipos)}")

Rows after cleaning: 247


In [8]:
# Remove TOPLAM rows and drop Sıra column
all_ipos_overs = all_ipos_overs[all_ipos_overs["Şirket Adı"] != "TOPLAM"]
all_ipos_overs = all_ipos_overs.dropna(subset=["Şirket Adı"])
all_ipos_overs = all_ipos_overs.drop(columns=["Sıra              "])
print(f"Overs after cleaning: {len(all_ipos_overs)}")

Overs after cleaning: 117


In [9]:
print("=== Main columns ===")
print(all_ipos[["Borsa Kodu", "Şirket Ünvanı", "Halka Arz Oranı (%)",
                "Halka Arz Fiyatı (TL)", "Yeni Sermaye (Bin TL)",
                "Halka Arza Aracılık Eden Kurum",
                "Borsada İşlem Görme Tarihi"]].head())

=== Main columns ===
  Borsa Kodu                                      Şirket Ünvanı  \
0      HLGYO            HALK GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.   
1      SRVGY          SERVET GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.   
2      PGSUS                     PEGASUS HAVA TAŞIMACILIĞI A.Ş.   
3      ROYAL  ROYAL HALI İPLİK TEKSTİL MOBİLYA SANAYİ VE TİC...   
4       ODAS           ODAŞ ELEKTRİK ÜRETİM SANAYİ TİCARET A.Ş.   

   Halka Arz Oranı (%)  Halka Arz Fiyatı (TL)  Yeni Sermaye (Bin TL)  \
0                28.00                   1.35               662500.0   
1                25.00                   2.73                52000.0   
2                35.00                  18.40               102272.0   
3                28.75                   4.45                60000.0   
4                28.57                   5.00                42000.0   

     Halka Arza Aracılık Eden Kurum Borsada İşlem Görme Tarihi  
0  Halk Yatırım Menkul Değerer A.Ş.                 2013-02-22  
1        Bizi

In [10]:
kap_sectors = pd.read_excel("KAP_Sectors.xlsx")
print(f"Sectors: {kap_sectors.shape}")
print(kap_sectors.columns.tolist())
print()
print(kap_sectors.head())

Sectors: (1454, 3)
['DİĞER', 'Unnamed: 1', 'Unnamed: 2']

                             DİĞER Unnamed: 1     Unnamed: 2
0                             Sıra        Kod  Şirket Unvanı
1                 Kayıt Bulunamadı        NaN            NaN
2                              NaN        NaN            NaN
3                              NaN        NaN            NaN
4  TARIM, ORMANCILIK VE BALIKÇILIK        NaN            NaN


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [11]:
# Parse KAP sector data
raw = pd.read_excel("KAP_Sectors.xlsx")
cols = raw.columns

sector_map = {}
current_sector = None

for _, row in raw.iterrows():
    val0 = str(row[cols[0]]).strip() if pd.notna(row[cols[0]]) else None
    val1 = row[cols[1]]

    # Skip empty rows, "Sıra" headers, "Kayıt Bulunamadı"
    if val0 is None or val0 == "nan":
        continue
    if val0 == "Sıra" or val0 == "Kayıt Bulunamadı":
        continue

    # If second column is empty, this row is a sector name
    if pd.isna(val1):
        current_sector = val0
    else:
        # This is a company row — val1 is the ticker
        ticker = str(val1).strip()
        sector_map[ticker] = current_sector

kap_df = pd.DataFrame(list(sector_map.items()), columns=["Borsa Kodu", "Sektör"])
print(f"Sectors mapped: {len(kap_df)} companies")
print(kap_df.head(10))

Sectors mapped: 606 companies
  Borsa Kodu                                             Sektör
0      AGROT  TARIM VE HAYVANCILIK AVCILIK VE İLGİLİ HİZMET ...
1      IZINV  TARIM VE HAYVANCILIK AVCILIK VE İLGİLİ HİZMET ...
2      KNFRT  TARIM VE HAYVANCILIK AVCILIK VE İLGİLİ HİZMET ...
3      OZSUB                          BALIKÇILIK VE SU ÜRÜNLERİ
4      YAPRK  TARIM VE HAYVANCILIK AVCILIK VE İLGİLİ HİZMET ...
5      CVKMD                          METAL CEVHERİ MADENCİLİĞİ
6      PRKME                        KÖMÜR VE LİNYİT MADENCİLİĞİ
7      RUZYE                        KÖMÜR VE LİNYİT MADENCİLİĞİ
8      TRMET                          METAL CEVHERİ MADENCİLİĞİ
9      TRENJ               HAM PETROL VE DOĞAL GAZ ÇIKARTILMASI


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [12]:
all_ipos = all_ipos.merge(kap_df, on="Borsa Kodu", how="left")

print(f"IPOs with sector: {all_ipos['Sektör'].notna().sum()}")
print(f"IPOs without sector: {all_ipos['Sektör'].isna().sum()}")

IPOs with sector: 236
IPOs without sector: 11


In [13]:
print(all_ipos[all_ipos["Sektör"].isna()][["Borsa Kodu", "Şirket Ünvanı"]])

    Borsa Kodu                                      Şirket Ünvanı
6        AKPAZ  AKYÜREK TÜKETİM ÜRÜNLERİ PAZARLAMA DAĞITIM VE ...
9        ARBUL             ARBUL ENTEGRE TEKSTİL İŞLETMELERİ A.Ş.
19       SNKRN  SENKRON SİBER GÜVENLİK YAZILIM VE BİLİŞİM ÇÖZÜ...
23       VIAGO             VİA GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.
68       OYYAT                  OYAK YATIRIM MENKUL DEĞERLER A.Ş.
76       KARYE            KARTAL YENİLENEBİLİR ENERJİ ÜRETİM A.Ş.
138       TERA                  TERA YATIRIM MENKUL DEĞERLER A.Ş.
153      GRTRK                               GRAINTURK TARIM A.Ş.
161      A1CAP            A1 CAPİTAL YATIRIM MENKUL DEĞERLER A.Ş.
189      SKYMD                 ŞEKER YATIRIM MENKUL DEĞERLER A.Ş.
217      EFORC                       EFOR ÇAY SANAYİ TİCARET A.Ş.


In [14]:
ticker_fixes = {
    "EFORC": "EFOR",
    "SNKRN": "GATEG",
    "KARYE": "A1YEN",
    "GRTRK": "GRTHO",
}

delisted = ["AKPAZ", "ARBUL", "VIAGO"]

for old, new in ticker_fixes.items():
    all_ipos.loc[all_ipos["Borsa Kodu"] == old, "Borsa Kodu"] = new
    match = kap_df[kap_df["Borsa Kodu"] == new]["Sektör"]
    if len(match) > 0:
        all_ipos.loc[all_ipos["Borsa Kodu"] == new, "Sektör"] = match.values[0]

# Check remaining missing sectors
missing = all_ipos[all_ipos["Sektör"].isna()][["Borsa Kodu", "Şirket Ünvanı"]]
print(f"Still missing sector: {len(missing)}")
print(missing)

Still missing sector: 7
    Borsa Kodu                                      Şirket Ünvanı
6        AKPAZ  AKYÜREK TÜKETİM ÜRÜNLERİ PAZARLAMA DAĞITIM VE ...
9        ARBUL             ARBUL ENTEGRE TEKSTİL İŞLETMELERİ A.Ş.
23       VIAGO             VİA GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.
68       OYYAT                  OYAK YATIRIM MENKUL DEĞERLER A.Ş.
138       TERA                  TERA YATIRIM MENKUL DEĞERLER A.Ş.
161      A1CAP            A1 CAPİTAL YATIRIM MENKUL DEĞERLER A.Ş.
189      SKYMD                 ŞEKER YATIRIM MENKUL DEĞERLER A.Ş.


In [15]:
manual_sectors = {
    "OYYAT": "ARACI KURUMLAR",
    "TERA": "ARACI KURUMLAR",
    "A1CAP": "ARACI KURUMLAR",
    "SKYMD": "ARACI KURUMLAR",
    "VIAGO": "GAYRİMENKUL YATIRIM ORTAKLIKLARI",
    "AKPAZ": "TOPTAN VE PERAKENDE TİCARET",
    "ARBUL": "TEKSTİL, GİYİM EŞYASI VE DERİ",
}

for ticker, sector in manual_sectors.items():
    all_ipos.loc[all_ipos["Borsa Kodu"] == ticker, "Sektör"] = sector

print(f"Still missing sector: {all_ipos['Sektör'].isna().sum()}")

Still missing sector: 0


In [16]:
all_ipos.to_csv("ipo_main_cleaned.csv", index=False)
all_ipos_overs.to_csv("ipo_overs_cleaned.csv", index=False)
print("Saved!")
print(f"Main: {len(all_ipos)} IPOs, {len(all_ipos.columns)} columns")
print(f"Overs: {len(all_ipos_overs)} IPOs, {len(all_ipos_overs.columns)} columns")

Saved!
Main: 247 IPOs, 18 columns
Overs: 117 IPOs, 18 columns


In [17]:
print(all_ipos.tail(5))
print()
# Check for rows where Borsa Kodu looks like a number or "Toplam" etc.
print(all_ipos[all_ipos["Borsa Kodu"].str.match(r'^\d+$', na=False)])

         Dönem Borsa Kodu                             Şirket Ünvanı  \
242   2025 / 9      DOFRB                   DOF ROBOTİK SANAYİ A.Ş.   
243  2025 / 11      ECOGR              ECOGREEN ENERJİ HOLDİNG A.Ş.   
244  2025 / 11      VAKFA                      VAKIF FAKTORİNG A.Ş.   
245  2025 / 11      PAHOL                      PASİFİK HOLDİNG A.Ş.   
246  2025 / 12      ZERGY  ZERAY GAYRİMENKUL YATIRIM ORTAKLIĞI A.Ş.   

                      Halka Arz Şekli  Halka Arz Oranı (%)  \
242   Sermaye artırımı ve ortak satış                29.03   
243                  Sermaye Artırımı                20.37   
244  Sermaye artırımı ve ortak satışı                25.00   
245  Sermaye artırımı ve ortak satışı                20.00   
246  Sermaye artırımı ve ortak satışı                25.05   

     Halka Arz Fiyatı (TL)  Ortak Satış Tutarı (Bin TL)  \
242                   45.0                      15000.0   
243                   10.4                          0.0   
244                   1

In [18]:
print(all_ipos[all_ipos["Halka Arz Fiyatı (TL)"].isna()])

Empty DataFrame
Columns: [Dönem, Borsa Kodu, Şirket Ünvanı, Halka Arz Şekli, Halka Arz Oranı (%), Halka Arz Fiyatı (TL), Ortak Satış Tutarı (Bin TL), Nakit Sermaye Artırım Tutarı (Bin TL), Satışa Hazır Bekletilen Pay Tutarı (Bin TL), Satışa Sunulan Nominal Toplam Tutar (Bin TL), Mevcut Sermaye (Bin TL), Yeni Sermaye (Bin TL), Satışa Sunulan Toplam Tutar (Piyasa Değeri Bin TL), Satışa Sunulan Toplam Tutar (Piyasa Değeri Bin ABD Doları), İlk İşlem Gördüğü Pazar, Halka Arza Aracılık Eden Kurum, Borsada İşlem Görme Tarihi, Sektör]
Index: []


In [19]:
print(all_ipos.groupby(all_ipos["Borsada İşlem Görme Tarihi"].dt.year).size())

Borsada İşlem Görme Tarihi
2013     9
2014     9
2015     4
2016     2
2017     3
2018     9
2019     6
2020     8
2021    52
2022    38
2023    56
2024    33
2025    18
dtype: int64
